# Lab 11 · Go keyless with Microsoft Entra ID and RBAC

Call your model through AI Gateway with a Microsoft Entra ID access token instead of an API key. You grant yourself
the **AI Gateway Models User** role on the gateway, inspect the token, call the model with the OpenAI SDK, and prove
that access follows the role assignment: `401` for the wrong audience, `403` without the role, `200` with it. This
notebook is the Python track of the [lab guide](README.md).

**Assign role → Get token → Call keyless → Wrong audience → Remove role → Restore**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| AI Gateway Models User (`a6b68e3d-a1f0-4c60-8678-237c050cf4ea`) | Data-plane role that lets you list and invoke models. Assignable at the gateway or a model alias. |
| Access token for `https://azure-api.net/aigateway` | Proves who you are. The gateway validates it at its edge, then checks your role assignments. |
| `get_bearer_token_provider` | Gives the OpenAI SDK a fresh token before every request, so no key is involved. |
| Alias `gpt-4.1-mini` | The model from notebook 02 that you call. |

| Role | Role ID | Grants |
| --- | --- | --- |
| AI Gateway User | `caad7a21-134e-43f4-a550-c5ffc17e051c` | List and invoke models, list and call MCP tools |
| AI Gateway Models User | `a6b68e3d-a1f0-4c60-8678-237c050cf4ea` | List and invoke models |
| AI Gateway Tools User | `dce03f8d-857f-4e40-a90d-67770fa6ecd2` | List and call tools within MCP servers |

![Identity boundaries: callers authenticate to the gateway with Entra ID, and the gateway authenticates to backends with its own identity](../assets/diagrams/identity-boundaries.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  reuses your CLI sign-in through `AzureCliCredential`.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-4.1-mini` alias.
- Permission to assign roles on the gateway, such as **Owner** or **Role Based Access Control Administrator**.

Role changes take a few minutes to apply, so this notebook waits and retries. Expect it to take 10 to 20 minutes.

Run the cells in order, or click **Run All**. Each step is idempotent: it reuses what already exists.

In [ ]:
%pip install -q --upgrade azure-identity requests openai

### 0. Initialize variables and clients

The defaults match notebooks 01 and 02. The cell also removes `AI_GATEWAY_API_KEY` from the environment to prove that
nothing in this notebook uses a key.

In [ ]:
import base64, hashlib, json, os, re, subprocess, sys, time, uuid
from pathlib import Path

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from openai import OpenAI, PermissionDeniedError

location       = os.environ.get("AIGW_LOCATION", "eastus2")                  # change to swedencentral if you prefer
resource_group = os.environ.get("AIGW_RESOURCE_GROUP", "rg-ai-gateway-labs") # change to use another resource group
gateway_name   = os.environ.get("AIGW_GATEWAY_NAME", "")                     # change to match notebook 01 (empty = derived)
alias          = "gpt-4.1-mini"                              # model alias from notebook 02
MODELS_USER    = "a6b68e3d-a1f0-4c60-8678-237c050cf4ea"      # AI Gateway Models User role ID
SCOPE          = "https://azure-api.net/aigateway/.default"  # AI Gateway token scope

os.environ.pop("AI_GATEWAY_API_KEY", None)   # nothing in this notebook uses a key

lab_name   = Path(globals().get("__vsc_ipynb_file__", "11-entra-rbac.ipynb")).stem
lab_labels = {"lab": lab_name, "workload": "ai-gateway-labs"}   # tags for clean-up

result = subprocess.run("az account show -o json", shell=True, capture_output=True, text=True)
if result.returncode != 0:
    sys.exit("❌ Azure CLI not installed or not signed in. Run `az login` first.")
account         = json.loads(result.stdout)
subscription_id = account["id"]
tenant_id       = account["tenantId"]
user_name       = account["user"]["name"]

suffix       = hashlib.sha1(subscription_id.encode()).hexdigest()   # stable per subscription
gateway_name = gateway_name or f"aigw-labs-{suffix[:6]}"
assert location in ("eastus2", "swedencentral"), "During the preview, AI Gateway runs in eastus2 or swedencentral."
assert re.fullmatch(r"[A-Za-z]([A-Za-z0-9-]{0,22}[A-Za-z0-9])?", gateway_name), "Use 1-24 letters, digits, or hyphens."

ARM        = "https://management.azure.com"
ARM_API    = "2025-09-01-preview"   # Microsoft.ApiManagement/aigateways
GATEWAY_ID = (f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
              f"/providers/Microsoft.ApiManagement/aigateways/{gateway_name}")

credential = AzureCliCredential()   # every client below reuses your az login
_arm_token = None

def arm(method, path, body=None, api_version=ARM_API, params=None):
    """Call Azure Resource Manager and return the JSON body (None when a GET or DELETE finds nothing)."""
    global _arm_token
    if _arm_token is None or _arm_token.expires_on - time.time() < 300:
        _arm_token = credential.get_token("https://management.azure.com/.default")
    response = requests.request(method, ARM + path, json=body, timeout=120,
                                params={"api-version": api_version, **(params or {})},
                                headers={"Authorization": f"Bearer {_arm_token.token}"})
    if response.status_code == 404 and method in ("GET", "DELETE"):
        return None
    if not response.ok:
        raise RuntimeError(f"{method} {path} -> {response.status_code}: {response.text[:1000]}")
    return response.json() if response.content else {}

def wait_until_ready(path, api_version=ARM_API, label="Resource", timeout_minutes=30):
    """Poll properties.provisioningState until it reports Succeeded."""
    start = time.time()
    while True:
        resource = arm("GET", path, api_version=api_version)
        state = "NotFound" if resource is None else resource.get("properties", {}).get("provisioningState", "Succeeded")
        if state == "Succeeded":
            return resource
        if state in ("Failed", "Canceled"):
            raise RuntimeError(f"{label} provisioning {state}: {json.dumps(resource.get('properties'))[:1000]}")
        if time.time() - start > timeout_minutes * 60:
            raise TimeoutError(f"{label} is still {state} after {timeout_minutes} minutes.")
        print(f"⏳ {label}: {state} ({int(time.time() - start)} s)")
        time.sleep(20)

gateway = arm("GET", GATEWAY_ID)
if not gateway or gateway["properties"].get("provisioningState") != "Succeeded":
    sys.exit(f"❌ Gateway {gateway_name} not found in {resource_group}. Run 01-provision-gateway.ipynb first.")
gateway_url = "https://" + gateway["properties"]["frontend"]["defaultHostname"]
WS          = f"{GATEWAY_ID}/workspaces/default"   # models, aliases, tool servers, and policies live here

def claims(token):
    """Decode the payload of a JWT access token. Never print the token itself."""
    payload = token.split(".")[1]
    return json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))

print(f"🧪 Lab:            {lab_name}")
print(f"👤 User:           {user_name}")
print(f"🏛️ Tenant:         {tenant_id}")
print(f"🔑 Subscription:   {account['name']} ({subscription_id})")
print(f"🌍 Region:         {location}")
print(f"📁 Resource group: {resource_group}")
print(f"🚪 Gateway:        {gateway_name}")
print(f"🌐 Runtime URL:    {gateway_url}")

### 1. Find your identity and check the model

Role assignments target an object ID. The cell reads yours from your Azure CLI token (the lab guide uses
`az ad signed-in-user show`), and confirms that notebook 02 published the alias.

In [ ]:
me             = claims(credential.get_token("https://management.azure.com/.default").token)
my_object_id   = me["oid"]
principal_type = "ServicePrincipal" if me.get("idtyp") == "app" else "User"
print(f"👤 {me.get('upn') or me.get('unique_name') or me.get('appid')}: {my_object_id} ({principal_type})")

aliases = arm("GET", f"{WS}/aliases")["value"]
if not any(a["properties"].get("aliasName") == alias for a in aliases):
    sys.exit(f"❌ Alias {alias} not found. Run 02-first-model-call.ipynb first.")
print(f"✅ Alias {alias} exists")

### 2. Grant yourself a data-plane role

Assign **AI Gateway Models User** to yourself on the gateway. These roles grant **runtime** access only, not gateway
administration. The assignment name is derived from the scope, role, and principal, so a rerun finds the same
assignment. The cell then lists your assignments that the gateway sees, including inherited ones.

In [ ]:
ROLE_API      = "2022-04-01"
ROLE_NAMES    = {MODELS_USER: "AI Gateway Models User",
                 "caad7a21-134e-43f4-a550-c5ffc17e051c": "AI Gateway User",
                 "dce03f8d-857f-4e40-a90d-67770fa6ecd2": "AI Gateway Tools User"}
ROLE_DEF_ID   = f"/subscriptions/{subscription_id}/providers/Microsoft.Authorization/roleDefinitions/{MODELS_USER}"
ASSIGNMENT_ID = (f"{GATEWAY_ID}/providers/Microsoft.Authorization/roleAssignments/"
                 f"{uuid.uuid5(uuid.NAMESPACE_URL, f'{GATEWAY_ID}|{MODELS_USER}|{my_object_id}'.lower())}")

def my_assignments():
    found = arm("GET", f"{GATEWAY_ID}/providers/Microsoft.Authorization/roleAssignments", api_version=ROLE_API,
                params={"$filter": f"principalId eq '{my_object_id}'"})["value"]
    return [a for a in found if a["properties"]["roleDefinitionId"].split("/")[-1] in ROLE_NAMES]

def assign_models_user():
    try:
        arm("PUT", ASSIGNMENT_ID, {"properties": {"roleDefinitionId": ROLE_DEF_ID, "principalId": my_object_id,
                                                  "principalType": principal_type}}, api_version=ROLE_API)
        return True
    except RuntimeError as error:
        if "RoleAssignmentExists" in str(error):
            return False
        raise

had_role_before = arm("GET", ASSIGNMENT_ID, api_version=ROLE_API) is not None or any(
    a["properties"]["roleDefinitionId"].endswith(MODELS_USER)
    and a["properties"]["scope"].lower() == GATEWAY_ID.lower() for a in my_assignments())
if had_role_before:
    print("♻️ You already have AI Gateway Models User on the gateway")
else:
    assign_models_user()
    print("✅ Assigned AI Gateway Models User on the gateway")

# The list can lag a new assignment by minutes, so read this notebook's assignment directly too
mine = {a["id"].lower(): a for a in my_assignments()}
created = arm("GET", ASSIGNMENT_ID, api_version=ROLE_API)
if created:
    mine[created["id"].lower()] = created
for a in mine.values():
    role = ROLE_NAMES[a["properties"]["roleDefinitionId"].split("/")[-1]]
    print(f"🔐 {role:<24} {a['properties']['scope']}")

### 3. Get a token and look inside it

Request a token for the AI Gateway scope. Apps that run as a service principal, managed identity, or agent identity
use `/.default`, and it works for an Azure CLI sign-in too. The cell decodes the payload to show what the gateway
checks: `aud` is `https://azure-api.net/aigateway`, and `tid` plus `oid` identify you to per-caller policies such as
the token limits in notebook 06.

In [ ]:
token  = credential.get_token(SCOPE).token
fields = {k: claims(token).get(k) for k in ("aud", "tid", "oid", "scp", "appid")}
for name, value in fields.items():
    print(f"🎫 {name:<6} {value}")
assert fields["aud"] == "https://azure-api.net/aigateway" and fields["oid"] == my_object_id

### 4. Call the model without a key

`get_bearer_token_provider` returns a cached token and refreshes it before it expires. The OpenAI SDK calls it before
every request and sends `Authorization: Bearer <token>`. This is what [infra/call_with_entra.py](infra/call_with_entra.py)
does, with `DefaultAzureCredential` instead of `AzureCliCredential`.

A new role assignment takes a few minutes to apply, so the cell retries `403` for up to 10 minutes.

In [ ]:
client = OpenAI(base_url=f"{gateway_url}/default/models/openai/v1",
                api_key=get_bearer_token_provider(credential, SCOPE))

deadline = time.time() + 600
while True:
    try:
        completion = client.chat.completions.create(model=alias, messages=[
            {"role": "user", "content": "In one sentence, why use Microsoft Entra ID instead of API keys?"}])
        break
    except PermissionDeniedError:
        if time.time() > deadline:
            raise
        print("⏳ HTTP 403, waiting for the role assignment to apply...")
        time.sleep(30)

print(f"📨 {completion.choices[0].message.content}")
print(f"🔢 Tokens used: {completion.usage.total_tokens}")
print("✅ Keyless call succeeded")

### 5. Send a token for the wrong audience

A token for Azure Resource Manager is valid, but not for AI Gateway. The gateway answers `401`: authentication
failed, so the role was never checked.

In [ ]:
chat_url = f"{gateway_url}/default/models/openai/v1/chat/completions"
payload  = {"model": alias, "max_tokens": 5, "messages": [{"role": "user", "content": "ping"}]}

def call_with(token):
    return requests.post(chat_url, json=payload, headers={"Authorization": f"Bearer {token}"}, timeout=60).status_code

arm_status = call_with(credential.get_token("https://management.azure.com/.default").token)
print(f"📨 ARM token:        HTTP {arm_status}")
assert arm_status == 401
print(f"📨 AI Gateway token: HTTP {call_with(credential.get_token(SCOPE).token)}")

### 6. Remove the role and try again

Remove your **AI Gateway Models User** assignments on the gateway, then call again. Once the change applies, the
gateway answers `403`: you're authenticated, but no role allows you to invoke the model. In the playground, the same
error offers **Review access** and **Copy access request**.

The cell polls for up to 10 minutes. If you also hold a role at a broader scope, such as the resource group, the call
keeps working and the cell prints a warning instead.

In [ ]:
def wait_for(expected, minutes=10):
    """Call the model with a fresh token until it returns the expected status, or time runs out."""
    deadline = time.time() + minutes * 60
    while True:
        status = call_with(credential.get_token(SCOPE).token)
        if status == expected or time.time() > deadline:
            return status
        print(f"⏳ HTTP {status}, waiting for {expected}...")
        time.sleep(30)

to_remove = {ASSIGNMENT_ID} | {a["id"] for a in my_assignments()
                               if a["properties"]["roleDefinitionId"].endswith(MODELS_USER)
                               and a["properties"]["scope"].lower() == GATEWAY_ID.lower()}
for assignment_id in to_remove:
    arm("DELETE", assignment_id, api_version=ROLE_API)
print("🗑️ Removed your AI Gateway Models User assignment on the gateway")

status = wait_for(403)
print(f"📨 Without the role: HTTP {status}")
if status == 403:
    print("✅ The gateway denies callers without a role")
else:
    print("⚠️ Still allowed. Check for an inherited role in the list from step 2.")

### 7. Restore access

Assign the role again and wait until the call returns `200`.

In [ ]:
assign_models_user()
print("✅ Assigned AI Gateway Models User again")
status = wait_for(200)
print(f"📨 With the role: HTTP {status}")
assert status == 200

### 8. Clean up resources

Role assignments don't cost anything. The cell removes the assignment this notebook created, and keeps it if you had
the role before you started. Set `keep_role = True` to keep it for other Entra-based labs.

In [ ]:
keep_role = False   # change to True to keep your role assignment

if keep_role or had_role_before:
    print("ℹ️ Kept your AI Gateway Models User assignment on the gateway")
else:
    arm("DELETE", ASSIGNMENT_ID, api_version=ROLE_API)
    print("🗑️ Removed your AI Gateway Models User assignment")

## Troubleshooting

- **`401` on every Entra call**: the token is expired, or it's for another audience, such as Azure Resource Manager.
  Request a token for `https://azure-api.net/aigateway/.default` and check `aud` in step 3.
- **`403` right after you assign the role**: role assignments take several minutes to apply. Wait and retry.
- **`400`**: the request sent both an `Authorization` header and an `api-key` header. Send one or the other.
- **`AuthorizationFailed` when you assign the role**: you need Owner, User Access Administrator, or Role Based Access
  Control Administrator on the gateway.
- **`200` after you removed the role**: you hold a role at a broader scope, such as the resource group or
  subscription. List them with `az role assignment list --assignee <object-id> --all -o table`.
- **`CredentialUnavailableError`**: run `az login`, then restart the kernel.

## Next steps

- Secure the outbound boundary in [Call backends with managed identities](../12-backend-identity/README.md).
- Budget each Entra identity in [Enforce cost budgets with per-caller overrides](../07-cost-budgets/README.md).
- Docs: [Security and identity](https://ai.gateway.azure.com/docs/security-identity).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/11-entra-rbac/](https://azure.github.io/ai-gateway/labs/11-entra-rbac/).